<a href="https://colab.research.google.com/github/Multiomics-Analytics-Group/course_multi-omics_analysis/blob/main/notebooks/05_Visualising_Networks/03_nx.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🕸️ Introduction to Networks in Python

**Day 3 · 9:30–10:00 · Multi-omics Data Science**

---

Yesterday ended with a list: the serum proteins that separate our three groups of septic
patients. A list is a set of independent facts — this protein is up, that one is down — and
it says nothing about how those facts hang together. A **network** does. A network is a
claim that the things we measured are *related*, and that the shape of those relations is
itself worth looking at.

We build that machinery from the bottom. The first half of the session uses graphs small
enough to draw on paper — three nodes, two edges — because every idea we need (neighbours,
degree, attributes, layouts) is easier to see there than in a hairball. The second half
takes the protein matrix from Day 2 — **1 458 protein groups measured in 45 septic
patients**, 15 per group — turns it into an edge list, and builds a co-abundance network we
can cluster, dissect and explore interactively.

The tool is [**NetworkX**](https://networkx.org/): a graph is an ordinary Python object,
nodes and edges can be anything hashable, and attributes are just dictionaries. It covers
construction, a large library of graph algorithms, drawing, and interchange with other
network software — including Cytoscape, which we meet at 11:00.

### What you will be able to do afterwards

1. Create a graph, add nodes and edges, and attach attributes to either.
2. Read a graph's basic structure — neighbours, degree, density, connected components.
3. Draw a network with several layouts, and explain why the layout is not part of the
   result.
4. Build a network from a real omics matrix, and defend the correlation cut-off you chose.
5. Detect communities with Louvain, extract one as a subgraph, and treat the partition as
   an algorithm's opinion rather than a fact.
6. Publish an interactive version of the network with PyVis.

## 1. Why a network at all

A network — a **graph**, in the mathematics — is two things: a set of **nodes** and a rule
for putting **edges** between them. That is the whole definition, and it is deceptively
generous: nodes can be proteins, patients, metabolites, papers or people, and the edge rule
can be anything you can compute. Everything interesting about a biological network is
hidden in that second half. *The rule you choose to draw edges is the model.*

A network is worth building when the **structure is the finding** — when the answer is not
"these twelve proteins changed" but "these twelve proteins changed *together*, and three of
them sit between two otherwise unconnected blocks". Degree, communities, hubs and shortest
paths are all statements about structure, and none of them can be made about a list. If
what you want to report is the node list, you do not need a network; you need a table.

NetworkX gives us the objects and the algorithms. It deliberately does *not* give us the
rule for drawing edges, or the meaning of the result — those stay our responsibility, and
they are what the rest of this notebook is really about.

## 2. Setup

Two extra libraries beyond NetworkX: `python-louvain` (imported as `community`) for
community detection, and `pyvis` for the interactive views at the end. Neither is part of a
standard Colab image, so we install them here. (`pyvis` is installed a second time in
section 11, where it is first used — a harmless no-op if you have run this cell, and a
convenience if you jumped straight there.)

We also fix where the course data lives. The helper below looks for a local copy first — so
the notebook works inside a clone of the repository — and falls back to reading the file
straight from GitHub, which is what happens on Colab. Nothing is downloaded until we ask
pandas to read it.

In [ ]:
# Extra libraries used in the second half of the notebook.
# On Colab (or a fresh environment) run this once; it is a no-op if they are already installed.
%pip install -q pyvis python-louvain

In [ ]:
import os

# Where the course data lives. The notebook works both inside a clone of the
# course repository and standalone (e.g. on Colab), where files are read from GitHub.
COURSE_REPO = "Multiomics-Analytics-Group/course_multi-omics_analysis"
BRANCH = "main"
BASE_URL = f"https://raw.githubusercontent.com/{COURSE_REPO}/{BRANCH}"


def course_file(relative_path):
    """Path to a course data file: a local copy if we are in the repo, otherwise the raw GitHub URL."""
    for prefix in ("", "..", "../..", "../../.."):
        candidate = os.path.join(prefix, relative_path) if prefix else relative_path
        if os.path.exists(candidate):
            return candidate
    return f"{BASE_URL}/{relative_path}"


PROTEOMICS_MATRIX = course_file("proteomics/data/protein_groups_matrix.tsv")
SAMPLE_METADATA = course_file("metadata/sample_metadata.tsv")
print(PROTEOMICS_MATRIX)

## 3. Building a graph by hand

Before touching real data we build a graph one node at a time. This is not busywork: the
co-abundance network later has a few hundred edges and you cannot check it by eye, so the
habits you want — look at the node count, look at the degrees, ask whether the graph is
connected — have to be formed on something small enough to verify.

`nx.Graph()` creates an **empty, undirected, unweighted** graph: no nodes, no edges. Two
other classes matter. `nx.DiGraph` makes edges directional — right for regulation, where
"A activates B" is not the same statement as "B activates A". `nx.MultiGraph` allows several
edges between the same pair of nodes, which is how you keep "co-expressed" and "physically
interacts" as separate evidence rather than collapsing them. Our co-abundance network is a
plain `Graph`: correlation is symmetric, so direction would be an invention.

In [26]:
import networkx as nx

In [27]:
G = nx.Graph()

### Adding nodes

A node can be any hashable Python object — an integer, a string, a tuple. Here we use
integers because they are short; in the real network the nodes are gene symbols, which is
far more readable and costs nothing.

**A single node**

In [28]:
G.add_node(1)

**Several nodes at once (from a list)**

In [29]:
G.add_nodes_from([2, 3])

**Number of nodes**

In [30]:
G.number_of_nodes()

3

### Adding edges

Edges are the interesting half. Note one behaviour that saves and occasionally costs you
time: adding an edge to a node that does not exist yet **creates that node silently**. It
means you can build a whole graph from an edge list without declaring nodes first — which is
exactly what we do later — but it also means a typo in a gene symbol produces a new,
lonely node rather than an error. Always check the node count against what you expected.

**A single edge**

In [ ]:
G.add_edge(1, 2)

**Several edges at once (from a list)**

In [32]:
G.add_edges_from([(1, 2),(1, 3)])

**Number of edges**

The edge `(1, 2)` was added twice — once on its own and once in the list. An undirected
`Graph` stores each pair once, so re-adding an existing edge updates it rather than
duplicating it. That is usually what you want, and it is worth knowing before you count.

In [33]:
G.number_of_edges()

2

**List of adjacent nodes (neighbours)**

In [34]:
list(G.neighbors(1))

[2, 3]

**Number of adjacent nodes (degree)**

Degree is the simplest structural measure there is, and the most used: in a co-abundance
network a high-degree node is a protein that moves with many others — a **hub**. Hubs are
the first thing people report from a network, and the first thing to be sceptical about,
because they are also what an over-permissive edge rule produces most readily.

In [35]:
list(nx.degree(G))

[(1, 2), (2, 1), (3, 1)]

## 4. Node and edge attributes

A graph in NetworkX is not only a topology; every node and every edge carries a dictionary
you can put anything in. This is where the biology lives. A node can hold a gene symbol, a
log₂ fold change and a p-value from the Day 2 analysis; an edge can hold the correlation
that justified it. Once attributes are attached they travel with the graph — into layouts,
into PyVis colours and sizes, and into the Cytoscape export at 11:00 — so attaching them
early is what turns a picture into a figure that carries data.

### Node attributes

Calling `add_node` on a node that already exists does not duplicate it; it **updates its
attributes**. That is why the two cells below annotate nodes 1 and 2 rather than creating
new ones.

In [ ]:
G.add_node(1, name='ALB')

In [ ]:
G.add_node(2, name='HP')

**Show the attributes**

`data=True` is the switch that turns a bare list of nodes into a list of
`(node, attribute dictionary)` pairs. The same argument works on `G.edges()`.

In [ ]:
list(G.nodes(data=True))

### Edge attributes

The same mechanism on edges. `weight` is a special name by convention — many NetworkX
algorithms (shortest paths, weighted degree, Louvain) look for it and use it automatically,
so calling your correlation `weight` changes the behaviour of functions you have not read
yet. We keep ours in an attribute named `r` later on, and pass it explicitly when we mean
it.

In [ ]:
G.add_edge(1, 2, weight=5.7, color='blue')

In [ ]:
G.add_edges_from([(3, 4), (4, 5)], color='red', weight=2.)

In [ ]:
G.add_edges_from([(3, 1), (1, 5)], color='blue', weight=2.)

**List the edge attributes**

In [ ]:
list(G.edges(data=True))

## 5. Drawing — and why the picture is not the data

NetworkX draws with matplotlib. Drawing needs one thing the graph does not contain:
**coordinates**. A graph has no geometry — only nodes and which of them are joined — so
every drawing function invents positions with a *layout algorithm*, and the choice of
algorithm changes the picture completely without changing a single edge.

| Layout | What it does | Reasonable use |
|---|---|---|
| `spring` (default) | force simulation: edges pull, nodes repel | showing modular structure |
| `circular` | nodes evenly on a circle | small graphs; comparing two graphs fairly |
| `spectral` | positions from the eigenvectors of the Laplacian | making a split into two blocks visible |
| `kamada_kawai` | places nodes to match graph-theoretic distances | tidy figures of medium graphs |

> ⚙️ **Layouts are not data.** A spring layout is a randomly initialised physical
> simulation: run it twice on the same graph and you get two different pictures. No
> coordinate means anything, no distance on the page is a quantity, and "these two proteins
> are near each other in the plot" is not a result. Pass `seed=` so your figure is
> reproducible, and read the biology out of the *edges* — never out of the positions.

In [ ]:
%matplotlib inline

In [ ]:
import matplotlib.pyplot as plt

**Drawing with the default layout**

In [ ]:
nx.draw(G)

**Drawing with labels**

`nx.draw` hides labels by default; `nx.draw_networkx` shows them. On a graph of five nodes
labels are obviously right. On the 150-protein network below they overlap into grey mush,
which is why we draw the labels separately there and can turn them off.

In [ ]:
nx.draw_networkx(G)

**Drawing with a spectral layout**

In [ ]:
nx.draw_spectral(G)

**Drawing with a circular layout**

Compare these four pictures. Same nodes, same edges, four different impressions — which is
the argument of the callout above, made visually.

In [ ]:
nx.draw_circular(G)

### ✋ Exercise 1

Build a small graph from scratch and give it structure. Do it in the empty cells below,
one step per cell, and look at the object after each step rather than only at the end.

1. Create a network with 10 nodes (`path_graph`)

2. Connect the nodes so that every node has degree of at least 2 (`degree`)

3. Add `color` as an attribute of nodes 1 to 5

4. Add a weight to the edges between nodes (1,2), (2,3), (3,4)

5. Visualise the network with several layouts

Then answer one question that is not in the list: which of your layouts would you put in a
paper, and what would a reader wrongly conclude from each of the others?

## 6. Real data: the course protein matrix

From here on we stop inventing graphs and derive one from measurements. The data are the
same serum proteomics we analysed on Day 2: **45 septic patients**, 15 per group, measured
by diaPASEF on a timsTOF Pro and quantified with DIA-NN.

| Group | Samples | Meaning |
| --- | --- | --- |
| `Con` | `Con1`...`Con15` | sepsis with negative cultures |
| `CSKP` | `KP1`...`KP15` | carbapenem-**susceptible** *Klebsiella pneumoniae* sepsis |
| `CRKP` | `CRKP1`...`CRKP15` | carbapenem-**resistant** *K. pneumoniae* sepsis |

We will turn this matrix into a **protein co-abundance network**: nodes are protein groups,
and two proteins are joined when their abundances move together across the 45 patients.
That construction is the workhorse of this part of the course — the practical after the
coffee break repeats it in more depth, and the Multi-omics II session this afternoon
extends it to two omics layers at once.

### Reading the matrix with pandas

One `read_csv` with `sep='\t'`. Look at the shape and the first rows before anything else:
four annotation columns, then one column per sample.

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv(PROTEOMICS_MATRIX, sep='\t', header=0)
print(df.shape)
df.head()

## 7. From a protein matrix to an edge list

This is the step that decides everything that follows, so it is worth slowing down. The
matrix has four annotation columns (`protein_group`, `protein_names`, `genes`,
`description`) and then one column per sample: the 45 patients plus three `QC_pool`
injections, which are technical replicates and not patients — we drop them.

The recipe, and the decision inside each step:

1. keep the 45 patient columns and take `log2` — intensities are multiplicative, and
   correlating them raw would let the most abundant proteins dominate;
2. label each row with its gene symbol, falling back to the accession, because `ALB` is
   readable in a figure and `P02768` is not;
3. keep only protein groups **quantified in every patient**, then the **150 most variable**
   of those — a complete-case rule, so that no edge is an artefact of imputation, and a
   variance rule, because a protein that barely moves cannot co-move with anything;
4. transpose to *samples × proteins* and correlate every pair with `.corr()`;
5. keep the pairs with `abs(r) > 0.7`.

> 🧬 **A correlation network is not an interaction network.** This is the single most
> common misreading of the figures we are about to make. An edge here says: across these
> 45 patients, these two proteins rose and fell together. It does **not** say they touch,
> bind, or regulate one another. Serum proteins co-vary for many reasons — released from
> the same damaged tissue, driven by the same acute-phase response, or simply both tracking
> how ill the patient is. That last one is the big confounder in a cohort of septic
> patients: severity is a shared upstream driver that will happily correlate two proteins
> that have nothing to do with each other. Each edge is a hypothesis, not a mechanism.

⚠️ **The threshold is a decision, not a fact.** `THRESHOLD = 0.7` is a choice, and the
graph's density, its connected components, its communities and every centrality ranking you
will compute are all downstream of it. Lower it and the network fills in until everything is
a hub; raise it and it fragments into isolated pairs. Nothing warns you, because at every
setting the code runs and produces a plausible picture. Note also that we are testing
150 × 149 / 2 ≈ 11 000 pairs without any multiple-testing correction, on 45 samples — so
some of these edges are chance. Re-run this section with 0.6 and 0.8 and watch the "result"
move; that is the honest way to find out how much of your finding is yours and how much is
the cut-off's.

In [ ]:
ANNOTATION_COLS = ["protein_group", "protein_names", "genes", "description"]

# The 45 patient samples: everything that is not annotation and not a QC pool
samples = [c for c in df.columns
           if c not in ANNOTATION_COLS and not c.startswith("QC_pool")]
print(len(samples), "patient samples:", samples[:3], "...", samples[-3:])

# log2 intensities, one row per protein group, labelled with the gene symbol
# (some rows have no gene symbol, or several; we take the first one, or the accession)
labels = df["genes"].fillna(df["protein_group"]).str.split(";").str[0]

log_data = np.log2(df[samples])
log_data.index = labels.values
log_data = log_data[~log_data.index.duplicated()]
print("log2 matrix:", log_data.shape)
log_data.iloc[:5, :5]

Now the two filters. Watch how many protein groups survive the complete-case rule: the full
matrix is about 27 % missing, so requiring a value in all 45 patients is a severe filter and
it is not neutral — it keeps the abundant, well-measured serum proteins and discards the
low-abundance ones, which is a biased slice of the proteome. We accept that bias here in
exchange for correlations that are computed from measurements rather than from imputed
values.

In [ ]:
# 1) keep only protein groups quantified in every patient (no missing values)
complete = log_data[log_data.notna().sum(axis=1) == len(samples)]
print("complete protein groups:", complete.shape[0])

# 2) of those, keep the 150 most variable ones -- constant proteins carry no
#    information about co-abundance
most_variable = complete.var(axis=1).sort_values(ascending=False).index[:150]
matrix = complete.loc[most_variable].T          # samples x proteins
print("matrix for correlation (samples x proteins):", matrix.shape)
matrix.iloc[:5, :5]

The correlation matrix is symmetric with a diagonal of ones, so a little over half of it is
redundant. `np.triu(..., k=1)` keeps the strict upper triangle — each pair exactly once,
no self-edges — and `.stack()` turns that half-matrix into the long, three-column table
that NetworkX wants: node, node, weight.

In [ ]:
# Pearson correlation between every pair of proteins across the 45 patients
correlation = matrix.corr()

# Keep the upper triangle only, so each pair appears once, and turn it into a long table
upper = np.triu(np.ones(correlation.shape, dtype=bool), k=1)
pairs = correlation.where(upper).stack()
pairs.index.names = ["protein_1", "protein_2"]
pairs = pairs.rename("r").reset_index()

THRESHOLD = 0.7
edges = pairs[pairs["r"].abs() > THRESHOLD].copy()
print(f"{len(edges)} edges with |r| > {THRESHOLD}")
edges.head()

## 8. From an edge list to a graph

`nx.from_pandas_edgelist` is the bridge: give it a data frame, the two node columns and the
attributes to carry across, and it builds the graph — creating each node the first time it
appears in an edge. Note what that implies: **proteins with no surviving edge simply are not
in the graph.** Of the 150 we started with, only those that correlate strongly with at least
one other protein appear at all, so the node count is already a result of the threshold.

Two numbers to read straight away. **Density** is the fraction of possible edges that exist
— near 0 means a sparse, interpretable network, near 1 means a hairball in which every
centrality measure is meaningless. **Connected components** counts the separate pieces; a
co-abundance network almost always has one large component plus a scattering of small ones,
and the small ones are often the more specific signal.

In [ ]:
G = nx.from_pandas_edgelist(edges, "protein_1", "protein_2", edge_attr="r")
print(G)
print("density:", round(nx.density(G), 4),
      "| connected components:", nx.number_connected_components(G))

## 9. Drawing the network

Now the picture — with the warning from section 5 doing real work. `spring_layout` is a
force simulation seeded from random positions, so `seed=42` is not decoration: without it,
every re-run of this cell gives a different figure of the *same graph*, and you would have
no way to compare today's plot with yesterday's. Nothing about where a node lands is a
measurement. Read the clumps, not the coordinates.

In [ ]:
plt.figure(figsize=(11, 9))
pos = nx.spring_layout(G, seed=42)
nx.draw(G, pos, node_size=180, node_color="#8ecae6", edge_color="#cccccc", with_labels=False)
nx.draw_networkx_labels(G, pos, font_size=7)
plt.title("Protein co-abundance network (|r| > 0.7)")
plt.axis("off")
plt.show()

## 10. Communities

Communities (or modules) are subsets of nodes with a higher density of connections among
themselves than with the rest of the network. Finding them is how you get from "here is a
hairball" to "here are five blocks of proteins that behave as units", and in a co-abundance
network a community is a group of proteins that moves as one block across the patients —
often a complex, a pathway, or proteins released together from the same tissue.

We use the **Louvain method**, the standard choice for large networks. It optimises
**modularity**: a score comparing the density of edges inside a community with the density
you would expect if the same nodes were wired at random. Louvain works greedily and in
passes — every node starts in its own community, nodes are moved one at a time to whichever
neighbouring community increases modularity most, then each community is collapsed to a
single node and the whole thing repeats on the smaller graph.
See the [Louvain method](https://en.wikipedia.org/wiki/Louvain_method) for the details.

> ⚙️ **A partition is an algorithm's opinion, not a property of the data.** Louvain is
> greedy and order-dependent, so different runs of the same graph can return different
> partitions; it has a `resolution` parameter that tunes how many communities you get; and
> it will happily partition a random graph, which has no communities at all. Modularity
> optimisation also suffers a known **resolution limit** — communities smaller than a scale
> set by the size of the whole network get absorbed into their neighbours. Treat the
> community a protein lands in as a hypothesis about its neighbours, and check whether it
> survives a re-run, a different resolution, and a different correlation threshold.

In [ ]:
import community.community_louvain as cm

communities = cm.best_partition(G)
print(len(set(communities.values())), "communities found")

> **Note.** `community.best_partition` comes from the `python-louvain` package, which is an
> extra dependency. NetworkX itself now ships a Louvain implementation,
> `nx.community.louvain_communities(G, seed=42)`, which returns a list of sets instead of a
> node → community dictionary. We keep `python-louvain` here because that is the API used in
> the rest of the course material, but the built-in function is a dependency-free
> alternative — and it takes a `seed`, which makes the run reproducible.

### Turning the result into a data frame

`best_partition` returns a dictionary mapping each node to a community number. A data frame
is easier to count, sort and join back onto the protein annotation — and the sizes are the
first thing to look at. A partition that is one enormous community and a handful of pairs
has not told you much.

In [ ]:
communities_df = pd.DataFrame.from_dict(communities, orient="index")
communities_df.columns = ["community"]
print(communities_df["community"].value_counts().head())
communities_df.head()

### Picking one community

We take the largest, which is the usual starting point — but "largest" is a convenience,
not a criterion. A small, tight community of proteins you recognise is often the more
informative object, and it is the one you can actually check against the literature.

In [ ]:
# The largest community is usually the most interesting one to look at
community_id = communities_df["community"].value_counts().idxmax()
community_nodes = communities_df[communities_df["community"] == community_id].index.tolist()
print("community", community_id, "with", len(community_nodes), "proteins")

In [ ]:
community_nodes

### Extracting the community as a subgraph

`G.subgraph(nodes)` returns a **view** onto the original graph — the same node and edge
objects, filtered, not copied. It is cheap, and it means edits to the view would touch `G`.
When you want an independent object to modify, take `G.subgraph(nodes).copy()`.

In [ ]:
C = G.subgraph(community_nodes)

### Visualising the community

This is the payoff of splitting the network up: at this size the labels are legible, so you
can finally ask the biological question — do you recognise these gene symbols, and do they
belong together for a reason you can name?

In [ ]:
plt.figure(figsize=(8, 7))
nx.draw(C, with_labels=True, node_size=600, node_color="#ffb703",
        edge_color="#999999", font_size=8)
plt.title(f"Community {community_id} of the co-abundance network")
plt.axis("off")
plt.show()

## 11. Other libraries for network visualisation

A static matplotlib figure is the right output for a paper and the wrong one for
exploration: you cannot hover a node to find out what it is, and you cannot pull a cluster
apart to see inside it. Two tools fill that gap in this course — **PyVis** here, which
renders a graph as a self-contained interactive HTML page, and **Cytoscape** at 11:00, which
is a desktop application built for publication-quality network figures with data-driven
styling.

### PyVis

[PyVis](https://pyvis.readthedocs.io/en/latest/index.html) wraps the `vis.js` JavaScript
library. `Network.from_nx()` ingests a NetworkX graph directly, and `save_graph()` writes an
HTML file you can open on its own or embed in the notebook — with `cdn_resources='in_line'`
so the JavaScript is bundled into the file and it still works offline.

In [ ]:
!pip install pyvis

In [ ]:
from pyvis.network import Network
from IPython.display import display, HTML

In [ ]:
nt = Network('600px', '100%', notebook=True, cdn_resources='in_line')
nt.from_nx(G)
nt.save_graph("protein_coabundance.html")
HTML(filename="protein_coabundance.html")

### Using attributes

PyVis reads specific node attributes and turns them into visual properties, which is the
whole point of attaching them back in section 4: instead of styling a figure by hand, you
put the data on the graph and let the renderer map it.

| Attribute | What PyVis does with it |
| --- | --- |
| `label` | the text drawn on the node |
| `title` | the tooltip shown on hover |
| `group` | assigns a colour, one per distinct value |
| `size`, `value` | node radius — the obvious place for degree |
| `color` | an explicit colour, overriding `group` |
| `x`, `y` | fixed positions, if you do not want the physics engine to decide |

The demonstration below uses a `cycle_graph` with six named proteins in four groups, so you
can see the mapping without the real network's clutter.

In [ ]:
nx_graph = nx.cycle_graph(6)
nx_graph.nodes[0]['title'] = 'ALB'
nx_graph.nodes[0]['group'] = 4
nx_graph.nodes[1]['title'] = 'HP'
nx_graph.nodes[1]['group'] = 1
nx_graph.nodes[2]['title'] = 'APOA1'
nx_graph.nodes[2]['group'] = 1
nx_graph.nodes[3]['title'] = 'PKM'
nx_graph.nodes[3]['group'] = 2
nx_graph.nodes[4]['title'] = 'ENO1'
nx_graph.nodes[4]['group'] = 2
nx_graph.nodes[5]['title'] = 'TUBB'
nx_graph.nodes[5]['group'] = 3
nt = Network('500px', '500px', notebook=True, cdn_resources='in_line')
# populates the nodes and edges data structures
nt.from_nx(nx_graph)
nt.save_graph("attributes_graph.html")
HTML(filename="attributes_graph.html")

PyVis also lets you add **interactive options** to change some of the parameters of your
network on the fly. It is very useful when you are looking for the best parameters to
display your network — and it is honest about what a layout is, because you can watch the
physics engine rearrange the same graph into a completely different-looking picture while
you drag the sliders.

To switch these options on, use:

``` python
net.show_buttons(filter_=['nodes','edges', 'physics'])
```

The next cell puts everything together: the co-abundance network, with the Louvain community
as each node's `group` so PyVis colours the modules, the gene symbol as the `label`, and the
correlation on each edge's tooltip. Hover a node to see its community and degree; that is
the version worth sending to a collaborator.

In [ ]:
# The same co-abundance network, with the Louvain community as the node "group"
# (PyVis colours nodes by group) and the correlation shown on hover.
P = G.copy()
for node in P.nodes():
    P.nodes[node]['group'] = communities[node]
    P.nodes[node]['title'] = f"{node} (community {communities[node]}, degree {P.degree(node)})"
    P.nodes[node]['label'] = node
for u, v, data in P.edges(data=True):
    data['title'] = f"r = {data['r']:.2f}"

nt = Network('600px', '100%', notebook=True, cdn_resources='in_line')
nt.from_nx(P)
nt.show_buttons(filter_=['nodes', 'edges', 'physics'])
nt.save_graph("coabundance_communities.html")
HTML(filename="coabundance_communities.html")

### ✋ Exercise 2

Build a network yourself and use node attributes to change how it looks.

Suggested route, using the sample metadata:

1. Read the patient groups with `metadata = pd.read_csv(SAMPLE_METADATA, sep="\t")` and
   look at the `sample_id` / `group` columns (`Con`, `CSKP`, `CRKP`).
2. Rebuild the co-abundance network **within one group only** (15 samples), for example the
   `CRKP` patients, and compare it with the `Con` network: how many edges survive the same
   `|r| > 0.7` threshold? Are the hub proteins the same?
3. Colour the nodes by Louvain community, and set `size` from the node degree.
4. Remember that with only 15 samples per group a correlation of 0.7 is much easier to reach
   by chance than with 45 — so a denser per-group network is **not** automatically a more
   biological one. Before you interpret any difference between the two graphs, decide what
   you would have to see to be convinced it is not the sample size talking.

## 🔗 Where this goes next

After the coffee break we do this again, properly. The co-abundance practical
[`04_nxpandas.ipynb`](https://colab.research.google.com/github/Multiomics-Analytics-Group/course_multi-omics_analysis/blob/main/notebooks/05_Visualising_Networks/04_nxpandas.ipynb) rebuilds the same network with more care — a
proper exploratory pass over the matrix, centrality measures rather than degree alone, node
attributes carried over from the Day 2 differential-abundance results — and exports it for
**Cytoscape** ([`material/cytoscape.md`](https://github.com/Multiomics-Analytics-Group/course_multi-omics_analysis/blob/main/material/cytoscape.md))
for publication-quality network figures. 
[Multi-omics II](https://colab.research.google.com/github/Multiomics-Analytics-Group/course_multi-omics_analysis/blob/main/multiomics/notebooks/02_multiomics_networks.ipynb) takes the same
construction across omics layers: proteins *and* metabolites as nodes in one network, where
the cross-layer edges are the interesting ones. All of it depends on the two habits from
this session — knowing what your edges mean, and knowing which of your conclusions are
really conclusions about a threshold.

## 📚 Further reading

- [NetworkX documentation](https://networkx.org/documentation/stable/) — the tutorial and
  the algorithm reference; start with the tutorial, then browse `nx.algorithms`.
- [PyVis documentation](https://pyvis.readthedocs.io/en/latest/index.html) — every node and
  edge option, and how to control the physics engine.
- Blondel VD *et al.* (2008) *Fast unfolding of communities in large networks.*
  J Stat Mech P10008. — the Louvain method itself; short, readable, and worth reading
  before you quote a community.
- Barabási A-L. *Network Science.* <http://networksciencebook.com/> — free online, the
  standard introduction to degree distributions, hubs and robustness.
- Shannon P *et al.* (2003) *Cytoscape: a software environment for integrated models of
  biomolecular interaction networks.* Genome Res 13:2498–2504. — the tool we use at 11:00.